# fasttransport

> Sync and async HTTP transports over httpx2, plus base classes for small REST clients

In [ ]:
#| hide
from cachy.core import enable_cachy, doms

In [ ]:
#| hide
enable_cachy(doms=doms+('jsonplaceholder.typicode.com',))

## Install

```sh
pip install fasttransport
```

## Why

Every small API wrapper needs the same plumbing: headers merged into each request, responses decoded by content type, errors that carry the response body, and SSE parsing for streams. Most wrappers also repeat the same mistake. They create one httpx client at startup and keep it, which pins its pooled connections to the event loop that created them. Code that later runs under a different loop, common in tests and background tasks, then fails with confusing connection errors.

fasttransport is that plumbing written once, on [httpx2](https://github.com/pydantic/httpx2). Each request gets a fresh client and nothing is tied to a loop. Pass `client=` to manage a persistent client yourself. [fastspec](https://github.com/AnswerDotAI/fastspec) builds its spec-driven clients on these transports, [ghapi](https://github.com/AnswerDotAI/ghapi) its GitHub client, and [solveit](https://github.com/AnswerDotAI/solveit) its gateway clients.

## Get started

`SyncTransport.request` executes a call and returns the decoded body: JSON arrives as a dict, text as text, and anything else as bytes. HTTP errors raise with the response body included in the message.

In [ ]:
from fasttransport.core import *

In [ ]:
SyncTransport().request('GET', 'https://jsonplaceholder.typicode.com/todos/1')

`AsyncTransport` is the same, awaited. It also has `stream`, an async generator that parses SSE events into JSON dicts. The [core page](https://AnswerDotAI.github.io/fasttransport/core.html) streams a chat completion with it.

In [ ]:
await AsyncTransport().request('GET', 'https://jsonplaceholder.typicode.com/users/1')

A small REST client is a base URL plus verbs. `HttpCli` verbs take a path relative to the base, kwargs become the payload (query params on `get` and `delete`, the JSON body otherwise), and decoded bodies come back:

In [ ]:
cli = HttpCli('https://jsonplaceholder.typicode.com')
cli.get('todos/1')

In [ ]:
cli.post('posts', title='hi')

`AsyncHttpCli` is the async twin. Its verbs return awaitables:

In [ ]:
acli = AsyncHttpCli('https://jsonplaceholder.typicode.com')
await acli.get('todos/1')

The [core page](https://AnswerDotAI.github.io/fasttransport/core.html) documents the rest with runnable examples: SSE streaming, error enrichment, multipart uploads, and managing your own client.